# Ocean Wave Height Prediction - Fixed Version

This notebook contains the corrected implementation for ocean wave height prediction using ConvLSTM.

## Key Fixes Applied:
1. Fixed channel dimension issue in dataset
2. Removed duplicate class definitions
3. Added proper debugging and validation


In [1]:
# Step 0: Environment Setup and Library Imports
# ------------------------------------------------------------------------------
import torch
torch.autograd.set_detect_anomaly(True)
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-GUI backend for training visualizations
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import os
import time
import pickle

warnings.filterwarnings('ignore')
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}")
    for i in range(num_gpus):
        print(f"GPU Name {i}: {torch.cuda.get_device_name(i)}")
    torch.cuda.empty_cache()

PyTorch Version: 2.5.1
Using device: cuda
Number of GPUs available: 2
GPU Name 0: NVIDIA GeForce GTX 1080 Ti
GPU Name 1: NVIDIA GeForce GTX 1080 Ti


In [2]:
# Step 1 & 2: Data Loading & Preprocessing
# ------------------------------------------------------------------------------
import numpy as np
import xarray as xr

training_data_path1 = r'dAtA\cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'
training_data_path_static = r'dAtA\cmems_mod_ibi_wav_my_0.027deg_static_1756186965833.nc'

print(f"\n--- Loading and casting data into RAM ---")

# FIX: Cast data to float32 to halve memory usage, then load into RAM.
ds_static = xr.open_dataset(training_data_path_static, engine="netcdf4").astype(np.float32).load()
ds1 = xr.open_dataset(training_data_path1, engine="netcdf4").astype(np.float32).load()
print("Data successfully loaded into memory as float32.")

static_mask = ds_static['mask'].values
static_depth = ds_static['deptho'].values

# Split data temporally
ds_train = ds1.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds1.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds1.sel(time=slice('2023-07-01', '2023-12-30'))

# Feature engineering
ds_train['VSDmag'] = np.sqrt(ds_train['VSDX']**2 + ds_train['VSDY']**2)
ds_val['VSDmag'] = np.sqrt(ds_val['VSDX']**2 + ds_val['VSDY']**2)
ds_test['VSDmag'] = np.sqrt(ds_test['VSDX']**2 + ds_test['VSDY']**2)

# Define variables
target_var = 'VCMX'
initial_feature_vars = ['VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
all_vars_to_process = [target_var] + initial_feature_vars

print(f"Target variable: {target_var}")
print(f"Feature variables: {initial_feature_vars}")
print(f"Number of feature variables: {len(initial_feature_vars)}")


--- Loading and casting data into RAM ---
Data successfully loaded into memory as float32.
Target variable: VCMX
Feature variables: ['VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
Number of feature variables: 8


In [ ]:
import pickle
import numpy as np
import os

print("--- Saving processed data to disk... ---")

# Create a directory to save the files
processed_data_dir = 'processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# 1. Save the scalers dictionary using pickle
with open(os.path.join(processed_data_dir, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)

# 2. Save the static data tensor numpy array
np.save(os.path.join(processed_data_dir, 'static_data_tensor.npy'), static_data_tensor)

# 3. Save the processed and split xarray datasets
# This is much faster because the data is already in memory
ds_train.to_netcdf(os.path.join(processed_data_dir, 'ds_train.nc'))
ds_val.to_netcdf(os.path.join(processed_data_dir, 'ds_val.nc'))
ds_test.to_netcdf(os.path.join(processed_data_dir, 'ds_test.nc'))

print(f"✅ All processed data has been saved to the '{processed_data_dir}' directory.")

In [4]:
# Step 2: Fit Scalers on Training Data Only
# ------------------------------------------------------------------------------
print("\n--- Fitting scalers on TRAINING data only... ---")
scalers = {}
chunk_size = 1000

for var in all_vars_to_process:
    if var == 'VMDR': 
        continue
    scaler = MinMaxScaler()
    for i in tqdm(range(0, ds_train.dims['time'], chunk_size), desc=f"Fitting scaler for {var}"):
        chunk = ds_train[var].isel(time=slice(i, i + chunk_size)).fillna(0).values.reshape(-1, 1)
        scaler.partial_fit(chunk)
    scalers[var] = scaler

# Process static data
static_scalers = {}
static_scalers['deptho'] = MinMaxScaler()
static_depth_scaled = static_scalers['deptho'].fit_transform(static_depth.reshape(-1, 1)).reshape(static_depth.shape)
static_data_tensor = np.stack([static_depth_scaled], axis=0).astype(np.float32)

print("Scalers fitted successfully.")
print(f"Static data tensor shape: {static_data_tensor.shape}")


--- Fitting scalers on TRAINING data only... ---


Fitting scaler for VCMX:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VSDmag:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VTM10:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VTM02:   0%|          | 0/27 [00:00<?, ?it/s]

...

Fitting scaler for VTM01_SW1:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VMXL:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VHM0_WW:   0%|          | 0/27 [00:00<?, ?it/s]

Fitting scaler for VHM0_SW1:   0%|          | 0/27 [00:00<?, ?it/s]

Scalers fitted successfully.
Static data tensor shape: (1, 72, 90)


In [ ]:
# Step 3: PyTorch Dataset (FIXED VERSION)
# ------------------------------------------------------------------------------
class WaveDataset_Lazy(Dataset):
    def __init__(self, ds, feature_vars, target_var, scalers, static_data_tensor, lookback=48, forecast_horizon=168):
        self.ds = ds
        self.feature_vars = feature_vars
        self.target_var = target_var
        self.scalers = scalers
        self.static_data_tensor = static_data_tensor
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        self.ds_length = len(self.ds['time'])
        self.n_sequences = self.ds_length - (self.lookback + self.forecast_horizon)
        self.all_vars = [target_var] + feature_vars

    def __len__(self):
        return self.n_sequences

    def __getitem__(self, idx):
        seq_start_idx = idx
        seq_end_idx = seq_start_idx + self.lookback + self.forecast_horizon
        data_slice = self.ds.isel(time=slice(seq_start_idx, seq_end_idx))
        data_slice = data_slice.fillna(0)

        # Apply scaling
        for var, scaler in self.scalers.items():
            if var not in data_slice.variables:
                continue
            var_data = data_slice[var].values
            original_shape = var_data.shape
            var_data_flattened = var_data.reshape(-1, 1)
            var_scaled_flattened = scaler.transform(var_data_flattened)
            var_scaled_reshaped = var_scaled_flattened.reshape(original_shape)
            data_slice[var] = (data_slice[var].dims, var_scaled_reshaped)

        # FIXED: Get time-varying features with correct channel dimension
        # Shape: (lookback, 1, lat, lon) for each feature
        features_list = [
            data_slice[var].isel(time=slice(0, self.lookback)).values[:, np.newaxis, :, :] 
            for var in self.feature_vars
        ]
        
        # Prepare static data with time dimension
        # Shape: (lookback, 2, lat, lon)
        static_data_with_time = np.tile(self.static_data_tensor[np.newaxis, ...], (self.lookback, 1, 1, 1))

        # Concatenate all features along channel dimension (axis=1)
        # Final shape: (lookback, total_channels, lat, lon)
        all_features = np.concatenate(features_list + [static_data_with_time], axis=1).astype(np.float32)

        # Target data
        y = data_slice[self.target_var].isel(time=slice(self.lookback, self.lookback + self.forecast_horizon)).values.astype(np.float32)

        return torch.from_numpy(all_features), torch.from_numpy(y)

# Create datasets and data loaders

lookback_hours = 48
forecast_hours = 168
batch_size = 8

train_dataset = WaveDataset_Lazy(ds_train, initial_feature_vars, target_var, scalers, static_data_tensor, 
                                lookback=lookback_hours, forecast_horizon=forecast_hours)
val_dataset = WaveDataset_Lazy(ds_val, initial_feature_vars, target_var, scalers, static_data_tensor, 
                              lookback=lookback_hours, forecast_horizon=forecast_hours)
test_dataset = WaveDataset_Lazy(ds_test, initial_feature_vars, target_var, scalers, static_data_tensor, 
                               lookback=lookback_hours, forecast_horizon=forecast_hours)

# FIX: Enabled parallel data loading to feed the GPU without bottlenecks.
# -------------------------------------------------------------------
num_cores = os.cpu_count() or 1 # Use 1 core if os.cpu_count() is None
print(f"Using {num_cores} CPU cores for parallel data loading.")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_cores, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_cores, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_cores, pin_memory=True)
# -------------------------------------------------------------------

print("DataLoaders instantiated successfully.")

# Debug: Check data shapes
print("\n--- Debugging Data Shapes ---")
sample_X, sample_y = next(iter(train_loader))
print(f"Sample X shape: {sample_X.shape}")
print(f"Sample y shape: {sample_y.shape}")

# Calculate expected input channels
n_feature_channels = len(initial_feature_vars)  # 8
n_static_channels = static_data_tensor.shape[0]  # 2 (mask + depth)
input_channels = n_feature_channels + n_static_channels  # 10
print(f"Expected input channels: {input_channels}")
print(f"Actual input channels from loader: {sample_X.shape[2]}") # Note: index is 2 now B, S, C, H, W

assert sample_X.shape[2] == input_channels, f"Channel mismatch! Expected {input_channels}, got {sample_X.shape[2]}"
print("✅ Channel dimensions are correct!")

Using 40 CPU cores for parallel data loading.
DataLoaders instantiated successfully.

--- Debugging Data Shapes ---


In [13]:
# Step 4: ConvLSTM Model Definition (CLEAN VERSION)
# ------------------------------------------------------------------------------

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2
        self.bias = bias
        self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, 
                             self.kernel_size, padding=self.padding, bias=self.bias)
    
    def forward(self, x, h_c):
        h, c = h_c
        combined = torch.cat([x, h], dim=1)
        cc = self.conv(combined)
        i, f, o, g = torch.split(cc, self.hidden_dim, dim=1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_n = f * c + i * g
        h_n = o * torch.tanh(c_n)
        return h_n, c_n
    
    def init_hidden(self, b, h, w, d):
        return (torch.zeros(b, self.hidden_dim, h, w, device=d), 
                torch.zeros(b, self.hidden_dim, h, w, device=d))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__()
        self.batch_first = batch_first
        self.num_layers = num_layers
        
        # Ensure hidden_dim is a list for multi-layer support
        hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim
        
        cell_list = []
        for i in range(self.num_layers):
            current_input_dim = input_dim if i == 0 else hidden_dims[i - 1]
            cell_list.append(ConvLSTMCell(current_input_dim, hidden_dims[i], kernel_size, bias))
        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, x, h_c=None):
        if not self.batch_first:
            x = x.permute(1, 0, 2, 3, 4)
        
        b, s_l, _, h, w = x.size()
        h_c = self._init_hidden(b, h, w, x.device) if h_c is None else h_c
        
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]
            output_inner = []
            for t in range(s_l):
                h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c])
                output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
        
        return cur_in, [h, c]

    def _init_hidden(self, b, h, w, d):
        init_states = []
        for cell in self.cell_list:
            init_states.append(cell.init_hidden(b, h, w, d))
        return init_states

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__()
        self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True)
        self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True)
        self.output_conv = nn.Conv2d(hidden_dims[1], 1, kernel_size=(1, 1), padding='same')

    def forward(self, x_seq):
        # The input x_seq already contains the concatenated static data
        l1_o, _ = self.cl1(x_seq)
        l2_o, _ = self.cl2(l1_o)
        
        return self.output_conv(l2_o[:, -1, :, :, :])

# Create model
print("\n--- Step 4: Building the ConvLSTM Model ---")
model = ConvLSTMNet(input_dim=input_channels)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
print(f"Model built and enabled for multi-GPU. Total Input Channels: {input_channels}")

# Test model with sample data
print("\n--- Testing Model Forward Pass ---")
model.eval()
with torch.no_grad():
    test_output = model(sample_X.to(device))
    print(f"Model output shape: {test_output.shape}")
    print("✅ Model forward pass successful!")


--- Step 4: Building the ConvLSTM Model ---
Model built and enabled for multi-GPU. Total Input Channels: 9

--- Testing Model Forward Pass ---
Model output shape: torch.Size([16, 1, 72, 90])
✅ Model forward pass successful!


In [6]:
# Step 5: Training Functions
# ------------------------------------------------------------------------------

def weighted_mse_loss(output, target, weight_threshold=0.8):
    """Weighted MSE loss that gives higher weight to larger wave heights"""
    mse = (output - target)**2
    weights = torch.where(target > weight_threshold, 2.0, 1.0)
    return torch.mean(weights * mse)

def train_model(model, train_loader, val_loader, device, epochs=50, model_path='best_convlstm_model.pth'):
    """Train the ConvLSTM model with early stopping"""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = weighted_mse_loss
    best_val_loss = float('inf')
    scaler = torch.cuda.amp.GradScaler()
    patience = 5
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    print("\n--- Step 5: Starting Model Training with Early Stopping ---")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        total_train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        
        for X, y in train_pbar:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                # Predict the next single step based on the input sequence
                predicted_step = model(X)
                # Target is the first time step of the forecast horizon
                loss = loss_fn(predicted_step, y[:, 0, :, :])
            
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            total_train_loss += loss.item()
            train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        total_val_loss = 0.0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]")
        
        with torch.no_grad():
            for X, y in val_pbar:
                X, y = X.to(device), y.to(device)
                
                with torch.cuda.amp.autocast():
                    predicted_step = model(X)
                    loss = loss_fn(predicted_step, y[:, 0, :, :])
                
                total_val_loss += loss.item()
                val_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        
        avg_val_loss = total_val_loss / len(val_loader)
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
        
        # Early stopping and model saving
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_path)
            print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs")
        
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    # Load best model
    model.load_state_dict(torch.load(model_path))
    print(f"\n✅ Training completed. Best model loaded from {model_path}")
    
    return model, history

print("Training functions defined successfully.")

Training functions defined successfully.


In [14]:
# Step 6: Start Training
# ------------------------------------------------------------------------------

# Train the model
trained_model, training_history = train_model(model, train_loader, val_loader, device, epochs=50)

# Plot training history
plt.figure(figsize=(10, 6))
plt.plot(training_history['train_loss'], label='Training Loss')
plt.plot(training_history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Training completed successfully!")


--- Step 5: Starting Model Training with Early Stopping ---


Epoch 1/50 [Training]:   0%|          | 0/1631 [00:00<?, ?it/s]

RuntimeError: Function 'PowBackward0' returned nan values in its 0th output.

In [ ]:
# Step 7: Model Evaluation
# ------------------------------------------------------------------------------

def evaluate_model(model, test_loader, device, scalers, target_var):
    """Evaluate the trained model on test data"""
    model.eval()
    all_predictions = []
    all_targets = []
    
    print("\n--- Evaluating Model on Test Data ---")
    
    with torch.no_grad():
        for X, y in tqdm(test_loader, desc="Evaluating"):
            X, y = X.to(device), y.to(device)
            
            # Get predictions
            predictions = model(X)
            
            # Move to CPU and convert to numpy
            predictions_np = predictions.cpu().numpy()
            targets_np = y[:, 0, :, :].cpu().numpy()  # First time step
            
            all_predictions.append(predictions_np)
            all_targets.append(targets_np)
    
    # Concatenate all predictions and targets
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    # Inverse transform to get original scale
    target_scaler = scalers[target_var]
    
    # Reshape for inverse transform
    pred_flat = predictions.reshape(-1, 1)
    target_flat = targets.reshape(-1, 1)
    
    pred_original = target_scaler.inverse_transform(pred_flat).reshape(predictions.shape)
    target_original = target_scaler.inverse_transform(target_flat).reshape(targets.shape)
    
    # Calculate metrics
    mse = np.mean((pred_original - target_original)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred_original - target_original))
    
    print(f"\n📊 Test Results:")
    print(f"MSE: {mse:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE: {mae:.6f}")
    
    return pred_original, target_original, {'mse': mse, 'rmse': rmse, 'mae': mae}

# Evaluate the model
predictions, targets, metrics = evaluate_model(trained_model, test_loader, device, scalers, target_var)

print("\n✅ Model evaluation completed!")

In [ ]:
# Step 8: Visualization
# ------------------------------------------------------------------------------

def plot_predictions(predictions, targets, sample_idx=0, save_path='prediction_comparison.png'):
    """Plot prediction vs target for a sample"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Target
    im1 = axes[0].imshow(targets[sample_idx], cmap='viridis')
    axes[0].set_title('Target (Ground Truth)')
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    plt.colorbar(im1, ax=axes[0])
    
    # Prediction
    im2 = axes[1].imshow(predictions[sample_idx], cmap='viridis')
    axes[1].set_title('Prediction')
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    plt.colorbar(im2, ax=axes[1])
    
    # Difference
    diff = predictions[sample_idx] - targets[sample_idx]
    im3 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
    axes[2].set_title('Difference (Pred - Target)')
    axes[2].set_xlabel('Longitude')
    axes[2].set_ylabel('Latitude')
    plt.colorbar(im3, ax=axes[2])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot some sample predictions
plot_predictions(predictions, targets, sample_idx=0)
plot_predictions(predictions, targets, sample_idx=10, save_path='prediction_comparison_2.png')

print("\n✅ Visualization completed!")
print("\n🎉 Ocean Wave Height Prediction Pipeline Completed Successfully!")
print(f"\n📈 Final Test Metrics:")
print(f"   RMSE: {metrics['rmse']:.4f}")
print(f"   MAE:  {metrics['mae']:.4f}")